<a href="https://colab.research.google.com/github/Gui-Rocha06/Chat-bot26/blob/main/aula08New.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Aula 07 - Chatbot utilizando o Gemini

# Instalação das bibliotecas
! pip install -q --upgrade pip jedi
! pip install -q llama-index llama-index-readers-file llama-index-llms-google-genai llama-index-embeddings-google-genai

In [ ]:
!pip install nbformat==5.10.4 nbconvert==7.16.6

In [ ]:
import nbformat
import nbconvert

print("nbformat:", nbformat.__version__)
print("nbconvert:", nbconvert.__version__)

In [7]:
# Rode esta celula antes de usar o Gemini com LlamaIndex
%pip install -q google-generativeai
import nest_asyncio
nest_asyncio.apply()
import os
from google.colab import userdata
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings


# Pega a variável de ambiente
os.environ['gemini_chatbot'] = userdata.get('gemini_chatbot')
api = os.environ['gemini_chatbot']


In [8]:
# Cria a variável chamada llm
llm = GoogleGenAI(
    model='gemini-2.0-flash',
    api_key=api
)

Settings.llm=llm
#Settings.embed_model = embed_model

1) Envie arquivo para a base de conhecimento utilizando a técnica RAG

Cria uma pasta chamada documentos no Colab e envie seus arquivos para lá


In [ ]:
from google.colab import files
import os
os.makedirs("/contents/documentos",exist_ok=True)
print('Pasta criada em /content/documentos')

In [10]:
# leitura dos arquivos
from llama_index.core import SimpleDirectoryReader
documentos = SimpleDirectoryReader(input_dir='/content/documentos')

In [ ]:
docs = documentos.load_data()
print(f'Quantidade de documentos carregados {len(docs)}')

In [ ]:
# Exibindo os metadados do documento
print(docs[0].get_metadata_str())

In [ ]:
from llama_index.core import node_parser
# Quebrando o documento em pedaços (nodes)
# importando a biblioteca
from llama_index.core.node_parser import SentenceSplitter
node_parser=SentenceSplitter(chunk_size=1200) # tamanho da divisão
# Fazer a divisão do documento carregado com base no chunk size
nodes = node_parser.get_nodes_from_documents(docs,show_progress=True)
print(f'Quantidade de nodes gerados: {len(nodes)}')

In [ ]:
# Configurando o LLM Gemi e o modelo embeddings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

llm = GoogleGenAI(
    model = 'gemini-2.5-flash',
    api_key = api
)

embed_model = GoogleGenAIEmbedding(
    model = 'gemini-embedding-001',
    api_key = api
)

Settings.llm = llm
Settings.embed_model = embed_model
print('LLM e embeddings configurados')

2) Criando o indice vetorial, esse indice é criado sem o Chroma DB, diretamente em memória para simplificar a execução no Colab

In [ ]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex(nodes,embed_model=embed_model)
print('Indice criado com sucesso')

In [ ]:
# Realizando consultas no Chatbot
# query engine realiza consulta simples no documento
query_engine = index.as_query_engine(llm=llm,similarity_top_k=1)
Response = query_engine.query("Quais grãos estão disponiveis")
print(Response)


In [ ]:
query_engine = index.as_query_engine(llm,similarity_top_k=1)
response = query_engine.query("Qual o preço dos grãos")
print(response.response)

4) Modo Chat memória resumida

In [18]:
from llama_index.core.memory import ChatSummaryMemoryBuffer
memory = ChatSummaryMemoryBuffer(llm=llm, token_limit=256)
chat_engine = index.as_chat_engine(
    chat_mode='context',
    llm=llm,
    memory=memory,
    system_prompt=(
        'Você é especialista em cafés da loja Serenatto, uma loja online que vende '
        'grãos de café torrados. Sua função é tirar dúvidas de forma simpática e natural sobre os grãos disponíveis'
    )
)

In [ ]:
response = chat_engine.chat("Olá")
print(response.response)

In [ ]:
response = chat_engine.chat('qual o preço citado antes')
print(response.response)

In [ ]:
memory.get()

In [ ]:
# Reset do chat
chat_engine.reset()
print('Chat resetado')

In [ ]:
response = chat_engine.chat('Como fazer um bom UV no Maya')
print(response.response)

In [25]:
# Criando a interface para o chatbot

! pip install -q ipywidgets

In [28]:
import ipywidgets as widgets
from IPython.display import display
from google.genai.errors import ClientError


In [ ]:

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Se ClientError vier de alguma lib específica, mantenha o import correto.
# Exemplo:
# from google.api_core.exceptions import ClientError

# Área de saída do chat
chat_output = widgets.Output(
    layout={
        "border": "1px solid #ccc",
        "padding": "10px",
        "height": "400px",
        "overflow_y": "auto"
    }
)

# Área de entrada do chat
input_box = widgets.Text(
    placeholder="Digite sua pergunta sobre a Serenatto",
    description="Pergunta:",
    layout=widgets.Layout(width="80%")
)

# Botões
send_button = widgets.Button(
    description="Enviar",
    button_style="info"   # troquei de primary para info
)

clear_button = widgets.Button(
    description="Limpar",
    button_style="warning"
)

# Histórico simples
history = []

def responder(pergunta):
    try:
        response = chat_engine.chat(pergunta)
        return getattr(response, "response", str(response))

    except ClientError as e:
        erro = str(e)

        if "429" in erro or "RESOURCE_EXHAUSTED" in erro:
            try:
                retriever = index.as_retriever(similarity_top_k=1)
                nodes = retriever.retrieve(pergunta)

                if nodes:
                    partes = []
                    for i, node in enumerate(nodes, 1):
                        texto = getattr(node, "text", str(node))
                        partes.append(f"**Trecho {i}:**\n\n{texto[:500]}")

                    return (
                        "A API Gemini atingiu o limite no momento. "
                        "Encontrei estes trechos locais:\n\n"
                        + "\n\n---\n\n".join(partes)
                    )

                return "A API Gemini atingiu o limite e não encontrei trechos locais relevantes."

            except Exception as erro_retriever:
                return f"A API Gemini atingiu o limite e houve erro ao buscar trechos locais: {erro_retriever}"

        return f"Erro na API: {erro}"

    except Exception as e:
        return f"Erro inesperado: {e}"

# Função para renderizar o chat
def render_chat():
    with chat_output:
        clear_output(wait=True)
        display(Markdown("## Chatbot Serenatto"))

        if not history:
            display(Markdown("_Nenhuma mensagem ainda._"))

        for role, content in history:
            if role == "user":
                display(Markdown(f"**Você:** {content}"))
            else:
                display(Markdown(f"**Serenatto:** {content}"))

# Evento de envio
def on_send(_):
    pergunta = input_box.value.strip()
    if not pergunta:
        return

    history.append(("user", pergunta))
    resposta = responder(pergunta)
    history.append(("assistant", resposta))

    input_box.value = ""
    render_chat()

# Evento de limpar
def on_clear(_):
    history.clear()
    try:
        chat_engine.reset()
    except Exception:
        pass
    render_chat()

send_button.on_click(on_send)
clear_button.on_click(on_clear)
input_box.on_submit(lambda _: on_send(None))

display(widgets.VBox([
    chat_output,
    widgets.HBox([input_box, send_button, clear_button])
]))

render_chat()
